In [2]:
from google.colab import drive
drive.mount("/content/drive")
!git clone -q -b m2/models-train https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
import sys; sys.path.insert(0, "/content/FloodVerify/src")
CK = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT/checkpoints"

Mounted at /content/drive


In [3]:
import numpy as np
from floodsitrep.cv.eval import metrics as M
truth = np.array([0, 0, 1, 1, 2, 3, 4, 4])
pred  = np.array([0, 1, 1, 0, 2, 3, 4, 3])
cm = M.confusion(pred, truth)
print(cm)
s = M.summarise(cm)
print("localisation F1:", s["loc_f1"])
print("damage F1:", s["damage_f1"])
print("xView2-style:", s["xview2_pixel_score"])

[[1 1 0 0 0]
 [1 1 0 0 0]
 [0 0 1 0 0]
 [0 0 0 1 0]
 [0 0 0 1 1]]
localisation F1: 0.8333333333333334
damage F1: {'no_damage': 0.6666666666666666, 'minor': 1.0, 'major': 0.6666666666666666, 'destroyed': 0.6666666666666666}
xView2-style: 0.759090909090909


In [9]:
import json, pandas as pd
rows = [json.loads(l) for l in open(f"{CK}/smoke_m1_s0/metrics.jsonl")]
cm = np.array(rows[-1]["val_confusion"])
names = ["background", "no dmg", "minor", "major", "destroyed"]
print(pd.DataFrame(cm, index=["true " + n for n in names], columns=["pred " + n for n in names]))

                 pred background  pred no dmg  pred minor  pred major  \
true background         23517127       532911           0          45   
true no dmg               163499       303507           2          23   
true minor                 11508        34609           0           0   
true major                 18551        57288           0           8   
true destroyed              1618         2816           0           4   

                 pred destroyed  
true background            1573  
true no dmg                  22  
true minor                    0  
true major                   32  
true destroyed                0  


In [10]:
rows = [json.loads(l) for l in open(f"{CK}/m1_siamese_unet_s0/metrics.jsonl")]
df = pd.DataFrame(rows)
print(df[["epoch", "train_loss", "val_loss", "xview2_pixel_score", "loc_f1",
          "dmg_f1_no_damage", "dmg_f1_minor", "dmg_f1_major", "dmg_f1_destroyed"]].round(3).to_string())
best = df["xview2_pixel_score"].idxmax()
print("\nbest epoch:", int(df.loc[best, "epoch"]))
print(pd.DataFrame(np.array(df.loc[best, "val_confusion"]),
                   index=["true " + n for n in names], columns=["pred " + n for n in names]))

    epoch  train_loss  val_loss  xview2_pixel_score  loc_f1  dmg_f1_no_damage  dmg_f1_minor  dmg_f1_major  dmg_f1_destroyed
0       0       0.797     0.728               0.240   0.624             0.579         0.033         0.340             0.055
1       1       0.667     0.705               0.262   0.722             0.652         0.052         0.267             0.027
2       2       0.624     0.677               0.240   0.752             0.664         0.071         0.493             0.006
3       3       0.598     0.681               0.285   0.756             0.528         0.051         0.565             0.040
4       4       0.581     0.659               0.284   0.784             0.669         0.030         0.473             0.049
5       5       0.565     0.692               0.248   0.751             0.527         0.067         0.427             0.010
6       6       0.557     0.673               0.295   0.769             0.631         0.065         0.411             0.041
7       

In [11]:
for run in ["m1_siamese_unet_s0", "m1_siamese_unet_ce_s0", "m0_early_fusion_unet_s0"]:
    df = pd.DataFrame([json.loads(l) for l in open(f"{CK}/{run}/metrics.jsonl")])
    b = df.loc[df["xview2_pixel_score"].idxmax()]
    print(f"{run:28s} best epoch {int(b.epoch):2d}  score {b.xview2_pixel_score:.3f}  loc {b.loc_f1:.3f}  "
          f"minor {b.dmg_f1_minor:.3f}  major {b.dmg_f1_major:.3f}  destroyed {b.dmg_f1_destroyed:.3f}")

m1_siamese_unet_s0           best epoch  9  score 0.316  loc 0.794  minor 0.112  major 0.425  destroyed 0.043
m1_siamese_unet_ce_s0        best epoch 12  score 0.363  loc 0.782  minor 0.100  major 0.449  destroyed 0.124
m0_early_fusion_unet_s0      best epoch  9  score 0.286  loc 0.756  minor 0.080  major 0.507  destroyed 0.032


In [ ]:
# Check the GPU before spending 15 minutes copying data. Expect "Tesla T4".
!nvidia-smi --query-gpu=name,memory.total --format=csv

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
# Mount Drive and set paths. R = the Gate 2 run folder.
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
RUN = "m1_siamese_unet_ce_s0"
R = f"{CK}/{RUN}"
EV_OUT = f"{D}/evidence_predicted/{RUN}"   # predicted evidence for Member 4
SCHEMA = "/content/FloodVerify/evidence/schema/evidence.schema.json"
!ls "{R}"     # expect best.pt, config.yaml, last.pt, metrics.jsonl

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
best.pt  config.yaml  last.pt  metrics.jsonl


In [ ]:
# Code: clone straight onto the evaluation branch (it also contains the metrics and models commits).
# First step out of the old folder: deleting the folder the shell is standing in breaks every later command.
%cd /content
# pip install -e makes the package visible to the separate Python that `!python -m` starts.
!rm -rf /content/FloodVerify
!git clone -q -b m3/eval-export https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!git log --oneline -3
!pip install -q -e . --no-deps jsonschema
import sys; sys.path.insert(0, "/content/FloodVerify/src")

/content
/content/FloodVerify
a1f9090 (HEAD -> m3/eval-export, origin/m3/eval-export) Calibration (temperature, ECE), evaluate CLI, predicted evidence export with crop stitching
3478494 (origin/m2/models-train) M0/M1/M2 models, Dice+focal/CE losses, resumable AMP train.py, run configs, Colab notebook
c29eeb9 (origin/m3/eval-metrics) cv/eval/metrics: pixel-level per-class IoU/F1, xView2-style score, tile bootstrap
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [ ]:
# Data to local disk (10-15 min). Skipped if already there in this session.
import os
if not os.path.exists(f"{X}/processed/evidence_labels"):
    !mkdir -p /content/xbd/processed
    !tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
    !tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
    !cp -r "{D}/xbd/hold" /content/xbd/
    !cp -r "{D}/xbd/tier3" /content/xbd/
    !cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
    !unzip -q "{D}/xbd/processed.zip" -d /content/xbd/processed
!ls /content/xbd
!ls /content/xbd/processed/evidence_labels | wc -l    # expect 10148

hold  processed  rscc_flood_manifest.csv  test	tier3  train
10148


In [ ]:
!python -m floodsitrep.cv.eval.evaluate --checkpoint "{R}/best.pt" --xbd-root {X} --role val --fit-temperature
!ls "{R}"

m1_siamese_unet_ce_s0 on val (2184 tiles), PIXEL level, 95% CI over tiles:
  xview2_pixel_score   0.3633  [0.2853, 0.4079]
  loc_f1               0.7820  [0.7649, 0.7974]
  damage_f1_hm         0.1838  [0.0734, 0.2449]
  dmg_f1_no_damage     0.6675  [0.6315, 0.7025]
  dmg_f1_minor         0.1002  [0.0613, 0.1450]
  dmg_f1_major         0.4492  [0.3735, 0.5195]
  dmg_f1_destroyed     0.1240  [0.0253, 0.2516]
  miou                 0.3442  [0.3289, 0.3605]
  temperature 1.133: ECE 0.0041 -> 0.0046, NLL 0.0524 -> 0.0518
best.pt  config.yaml  eval_val.json  last.pt  metrics.jsonl  temperature.json


In [ ]:
# Read the saved results back (this is what goes into the report).
import json, pandas as pd, numpy as np
ev = json.load(open(f"{R}/eval_val.json"))
b = ev["bootstrap_95"]
keys = ["xview2_pixel_score", "loc_f1", "damage_f1_hm", "dmg_f1_no_damage", "dmg_f1_minor",
        "dmg_f1_major", "dmg_f1_destroyed", "miou"]
print(pd.DataFrame({k: b[k] for k in keys}).T.round(4))
t = json.load(open(f"{R}/temperature.json"))
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in t.items()
       if k in ("temperature", "temperature_at_search_bound", "ece_before", "ece_after", "nll_before", "nll_after", "pixels")})

                     value      lo      hi
xview2_pixel_score  0.3633  0.2853  0.4079
loc_f1              0.7820  0.7649  0.7974
damage_f1_hm        0.1838  0.0734  0.2449
dmg_f1_no_damage    0.6675  0.6315  0.7025
dmg_f1_minor        0.1002  0.0613  0.1450
dmg_f1_major        0.4492  0.3735  0.5195
dmg_f1_destroyed    0.1240  0.0253  0.2516
miou                0.3442  0.3289  0.3605
{'temperature_at_search_bound': False, 'pixels': 4321087, 'temperature': 1.1326, 'nll_before': 0.0524, 'nll_after': 0.0518, 'ece_before': 0.0041, 'ece_after': 0.0046}


In [ ]:
!python -m floodsitrep.cv.export_evidence --from-gt --xbd-root {X} --role val --out /content/ev_from_gt --schema {SCHEMA} --label-dir {X}/processed/evidence_labels

wrote 2184 evidence files to /content/ev_from_gt
{
  "tiles": 2184,
  "cells": 18997,
  "count_exact_match": 0.9667315891982945,
  "building_ratio_pred_over_label": 0.9187523347030258,
  "class_fraction_mae": 0.0006884540453816182,
  "cells_both_with_buildings": 4363,
  "quantifier_agreement": {
    "none": 0.9965784071169131,
    "minor": 0.9965784071169131,
    "major": 0.9964204874453861,
    "destroyed": 0.9965257672264042
  }
}


In [ ]:
for role in ("val", "test"):
    !python -m floodsitrep.cv.export_evidence --checkpoint "{R}/best.pt" --temperature "{R}/temperature.json" --xbd-root {X} --role {role} --out "{EV_OUT}" --schema {SCHEMA}
!ls "{EV_OUT}" | wc -l     # expect 2184 + 2476 = 4660

500 tiles written
1000 tiles written
1500 tiles written
2000 tiles written
wrote 2184 evidence files to /content/drive/MyDrive/MAAI 902 X 901 PROJECT/evidence_predicted/m1_siamese_unet_ce_s0
500 tiles written
1000 tiles written
1500 tiles written
2000 tiles written
wrote 2476 evidence files to /content/drive/MyDrive/MAAI 902 X 901 PROJECT/evidence_predicted/m1_siamese_unet_ce_s0
4660


In [ ]:
# Agreement of predicted evidence with label evidence (val tiles only: compare, never tune on test).
from floodsitrep.cv.export_evidence import compare_with_labels
val_ids = {f.replace(".json", "") for f in os.listdir(EV_OUT) if f.startswith("hurricane-florence")}
import shutil, pathlib
tmp = pathlib.Path("/content/ev_pred_val"); tmp.mkdir(exist_ok=True)
for i in val_ids:
    shutil.copy(f"{EV_OUT}/{i}.json", tmp / f"{i}.json")
print(json.dumps(compare_with_labels(tmp, f"{X}/processed/evidence_labels"), indent=2))

{
  "tiles": 2184,
  "cells": 18997,
  "count_exact_match": 0.852029267779123,
  "building_ratio_pred_over_label": 1.0721890175569668,
  "class_fraction_mae": 0.20894105913084776,
  "cells_both_with_buildings": 4211,
  "quantifier_agreement": {
    "none": 0.8846659998947202,
    "minor": 0.8651366005158709,
    "major": 0.901405485076591,
    "destroyed": 0.9312523029952098
  }
}


In [ ]:
# Look at one predicted file next to its label file.
tid = sorted(val_ids)[0]
P = json.load(open(f"{EV_OUT}/{tid}.json")); L = json.load(open(f"{X}/processed/evidence_labels/{tid}.json"))
rows = []
for p, l in zip(P["cells"], L["cells"]):
    rows.append({"cell": p["cell"], "label_buildings": l["buildings"], "pred_buildings": p["buildings"],
                 "label_destroyed": l["class_fraction"]["destroyed"], "pred_destroyed": p["class_fraction"]["destroyed"],
                 "pred_confidence": p["confidence"], "valid_fraction": p.get("valid_fraction")})
print(tid); print(pd.DataFrame(rows).to_string(index=False))

hurricane-florence_00000000_p1
cell  label_buildings  pred_buildings  label_destroyed  pred_destroyed  pred_confidence  valid_fraction
r0c0                5               4              0.0             0.0           0.5049             1.0
r0c1                6               5              0.0             0.0           0.5394             1.0
r0c2                0               0              0.0             0.0           0.5688             1.0
r1c0                4               2              0.0             0.0           0.3697             1.0
r1c1                4               2              0.0             0.0           0.5658             1.0
r1c2                1               1              0.0             0.0           0.6305             1.0
r2c0                1               1              0.0             0.0           0.6259             1.0
r2c1                0               0              0.0             0.0           0.9910             1.0
r2c2                1            

In [ ]:
from google.colab import runtime
runtime.unassign()

In [4]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

/bin/bash: line 1: nvidia-smi: command not found


In [5]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
!ls "{CK}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
m0_early_fusion_unet_s0  m1_siamese_unet_ce_s2	m1_siamese_unet_s2
m1_siamese_unet_ce_s0	 m1_siamese_unet_s0	smoke_m1_s0
m1_siamese_unet_ce_s1	 m1_siamese_unet_s1


In [6]:
%cd /content
!rm -rf /content/FloodVerify
!git clone -q -b m3/eval-export https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!git log --oneline -3
!pip install -q -e . --no-deps jsonschema
import sys; sys.path.insert(0, "/content/FloodVerify/src")

/content
/content/FloodVerify
a1f9090 (HEAD -> m3/eval-export, origin/m3/eval-export) Calibration (temperature, ECE), evaluate CLI, predicted evidence export with crop stitching
3478494 (origin/m2/models-train) M0/M1/M2 models, Dice+focal/CE losses, resumable AMP train.py, run configs, Colab notebook
c29eeb9 (origin/m3/eval-metrics) cv/eval/metrics: pixel-level per-class IoU/F1, xView2-style score, tile bootstrap
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [7]:
import os
if not os.path.exists(f"{X}/processed/evidence_labels"):
    !mkdir -p /content/xbd/processed
    !tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
    !tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
    !cp -r "{D}/xbd/hold" /content/xbd/
    !cp -r "{D}/xbd/tier3" /content/xbd/
    !cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
    !unzip -q "{D}/xbd/processed.zip" -d /content/xbd/processed
!ls /content/xbd
!ls /content/xbd/processed/evidence_labels | wc -l   # expect 10148

hold  processed  rscc_flood_manifest.csv  test	tier3  train
10148


In [5]:
!python -m floodsitrep.cv.train --config configs/cv/m1_siamese_unet_ce.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set seed=1

run m1_siamese_unet_ce_s1: train 5488 tiles, val 2184 tiles, device cuda, amp True
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100% 83.3M/83.3M [00:00<00:00, 195MB/s]
  epoch 0 step 50/686 loss 1.3441 (running 1.5567)
  epoch 0 step 100/686 loss 0.9477 (running 1.3885)
  epoch 0 step 150/686 loss 0.6151 (running 1.2180)
  epoch 0 step 200/686 loss 0.6822 (running 1.0883)
  epoch 0 step 250/686 loss 0.5160 (running 0.9956)
  epoch 0 step 300/686 loss 0.7415 (running 0.9278)
  epoch 0 step 350/686 loss 0.3895 (running 0.8760)
  epoch 0 step 400/686 loss 0.5831 (running 0.8428)
  epoch 0 step 450/686 loss 1.2408 (running 0.8180)
  epoch 0 step 500/686 loss 0.4697 (running 0.7930)
  epoch 0 step 550/686 loss 0.2436 (running 0.7668)
  epoch 0 step 600/686 loss 0.5604 (running 0.7459)
  epoch 0 step 650/686 loss 1.1494 (running 0.7255)
epoch 0: train_loss 0.7133 val_loss 0.1696 xview2_pixel 0.2486 loc_f1

In [7]:
!python -m floodsitrep.cv.train --config configs/cv/m1_siamese_unet.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set seed=1

run m1_siamese_unet_s1: train 5488 tiles, val 2184 tiles, device cuda, amp True
  epoch 0 step 50/686 loss 0.9771 (running 1.0376)
  epoch 0 step 100/686 loss 0.8649 (running 0.9928)
  epoch 0 step 150/686 loss 0.8473 (running 0.9508)
  epoch 0 step 200/686 loss 0.7943 (running 0.9157)
  epoch 0 step 250/686 loss 0.7556 (running 0.8911)
  epoch 0 step 300/686 loss 0.7711 (running 0.8741)
  epoch 0 step 350/686 loss 0.7666 (running 0.8594)
  epoch 0 step 400/686 loss 0.7538 (running 0.8489)
  epoch 0 step 450/686 loss 0.8516 (running 0.8406)
  epoch 0 step 500/686 loss 0.7519 (running 0.8311)
  epoch 0 step 550/686 loss 0.6511 (running 0.8229)
  epoch 0 step 600/686 loss 0.8276 (running 0.8156)
  epoch 0 step 650/686 loss 0.9648 (running 0.8086)
epoch 0: train_loss 0.8042 val_loss 0.7670 xview2_pixel 0.1392 loc_f1 0.4620 dmg_f1 minor 0.049 major 0.232 destroyed 0.000 (394.5s)
  epoch 1 step 50/686 loss 0.6527 (running 0.7186)
  epoch 1 step 100/686 loss 0.7032 (running 0.7209)
  epoch 1

In [8]:
!python -m floodsitrep.cv.train --config configs/cv/m1_siamese_unet_ce.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set seed=2

run m1_siamese_unet_ce_s2: train 5488 tiles, val 2184 tiles, device cuda, amp True
  epoch 0 step 50/686 loss 1.3990 (running 1.4616)
  epoch 0 step 100/686 loss 1.1290 (running 1.3390)
  epoch 0 step 150/686 loss 0.9754 (running 1.1927)
  epoch 0 step 200/686 loss 0.6796 (running 1.0797)
  epoch 0 step 250/686 loss 0.5313 (running 1.0000)
  epoch 0 step 300/686 loss 0.8585 (running 0.9319)
  epoch 0 step 350/686 loss 0.2325 (running 0.8814)
  epoch 0 step 400/686 loss 0.5114 (running 0.8450)
  epoch 0 step 450/686 loss 0.1984 (running 0.8206)
  epoch 0 step 500/686 loss 0.3034 (running 0.7942)
  epoch 0 step 550/686 loss 0.5663 (running 0.7658)
  epoch 0 step 600/686 loss 0.8423 (running 0.7459)
  epoch 0 step 650/686 loss 0.3982 (running 0.7312)
epoch 0: train_loss 0.7179 val_loss 0.1957 xview2_pixel 0.1973 loc_f1 0.6577 dmg_f1 minor 0.000 major 0.001 destroyed 0.000 (393.5s)
  epoch 1 step 50/686 loss 0.6121 (running 0.5672)
  epoch 1 step 100/686 loss 0.4137 (running 0.5059)
  epoc

In [9]:
!python -m floodsitrep.cv.train --config configs/cv/m1_siamese_unet.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set seed=2

run m1_siamese_unet_s2: train 5488 tiles, val 2184 tiles, device cuda, amp True
  epoch 0 step 50/686 loss 0.9980 (running 1.0163)
  epoch 0 step 100/686 loss 0.9337 (running 0.9854)
  epoch 0 step 150/686 loss 0.8846 (running 0.9513)
  epoch 0 step 200/686 loss 0.8149 (running 0.9244)
  epoch 0 step 250/686 loss 0.7663 (running 0.9033)
  epoch 0 step 300/686 loss 0.8324 (running 0.8834)
  epoch 0 step 350/686 loss 0.8374 (running 0.8693)
  epoch 0 step 400/686 loss 0.7709 (running 0.8565)
  epoch 0 step 450/686 loss 0.7329 (running 0.8477)
  epoch 0 step 500/686 loss 0.7328 (running 0.8405)
  epoch 0 step 550/686 loss 0.6720 (running 0.8320)
  epoch 0 step 600/686 loss 0.7437 (running 0.8244)
  epoch 0 step 650/686 loss 0.7543 (running 0.8201)
epoch 0: train_loss 0.8158 val_loss 0.7501 xview2_pixel 0.1432 loc_f1 0.4773 dmg_f1 minor 0.035 major 0.382 destroyed 0.000 (393.8s)
  epoch 1 step 50/686 loss 0.6812 (running 0.7329)
  epoch 1 step 100/686 loss 0.6894 (running 0.7251)
  epoch 1

In [10]:
from google.colab import runtime
runtime.unassign()

In [12]:
rows = []
for loss, cfg in [("Dice+focal", "m1_siamese_unet"), ("weighted CE", "m1_siamese_unet_ce")]:
    for seed in (0, 1, 2):
        run = f"{cfg}_s{seed}"
        df = pd.DataFrame([json.loads(l) for l in open(f"{CK}/{run}/metrics.jsonl")])
        b = df.loc[df["xview2_pixel_score"].idxmax()]
        rows.append({"loss": loss, "seed": seed, "best_epoch": int(b.epoch),
                     "score": b.xview2_pixel_score, "loc": b.loc_f1, "minor": b.dmg_f1_minor,
                     "major": b.dmg_f1_major, "destroyed": b.dmg_f1_destroyed})
t = pd.DataFrame(rows)
print(t.round(3).to_string(index=False))
print(t.groupby("loss")[["score", "loc", "minor", "major", "destroyed"]].agg(["mean", "std"]).round(3))

       loss  seed  best_epoch  score   loc  minor  major  destroyed
 Dice+focal     0           9  0.316 0.794  0.112  0.425      0.043
 Dice+focal     1          14  0.326 0.819  0.107  0.524      0.045
 Dice+focal     2          11  0.313 0.787  0.099  0.598      0.044
weighted CE     0          12  0.363 0.782  0.100  0.449      0.124
weighted CE     1          12  0.363 0.803  0.086  0.519      0.123
weighted CE     2          18  0.356 0.803  0.100  0.532      0.090
             score           loc         minor         major        destroyed  \
              mean    std   mean    std   mean    std   mean    std      mean   
loss                                                                            
Dice+focal   0.318  0.007  0.800  0.017  0.106  0.007  0.516  0.087     0.044   
weighted CE  0.361  0.004  0.796  0.012  0.096  0.008  0.500  0.044     0.113   

                    
               std  
loss                
Dice+focal   0.001  
weighted CE  0.019  
